In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. GPU 사용 가능 여부 확인 및 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 중인 디바이스: {device}")

# 2. 모델과 토크나이저 로드
base_model_name = "gogamza/kobart-base-v2"
model_path = "./model/full_finetuned_bidirectional_model_002"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

# 3. 양방향/복원 번역을 위한 토큰 정의 (학습 시 사용했던 토큰과 동일해야 함)
tokenizer.add_special_tokens({'additional_special_tokens': ['[제주]', '[표준]', '[복원]']})
model.resize_token_embeddings(len(tokenizer))

# 4. 추론 함수 정의
def predict(text):
    """
    주어진 텍스트를 모델로 추론합니다.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    if 'token_type_ids' in inputs:
        del inputs['token_type_ids']

    # 모델 추론
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=512
            num_beams=5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    # 결과 디코딩 및 반환
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# 5. 각 태스크별 성능 테스트
# 예시 1: '제주 방언 → 표준어' 번역
jeju_sentence = "[제주] 호끔만 이 시믄 오셩 부수지 맙서.."
translated_to_std = predict(jeju_sentence)
print(f"--- 제주 방언 → 표준어 번역 ---")
print(f"원문: {jeju_sentence}")
print(f"번역: {translated_to_std}\n")

# 예시 2: '표준어 → 제주 방언' 번역
std_sentence = "[표준] 좋은 아침입니다~ 여러분 주말 잘 쉬셨나요?"
translated_to_jeju = predict(std_sentence)
print(f"--- 표준어 → 제주 방언 번역 ---")
print(f"원문: {std_sentence}")
print(f"번역: {translated_to_jeju}\n")

# 예시 3: '복원' 태스크
restore_sentence = "[복원] 교육 지원을 받을때 차질이 없도록 진행해주시면 감사하겠습니다~"
restored_text = predict(restore_sentence)
print(f"--- 복원 태스크 ---")
print(f"원문: {restore_sentence}")
print(f"복원: {restored_text}\n")

사용 중인 디바이스: cpu


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


--- 제주 방언 → 표준어 번역 ---
원문: [제주] 호끔만 이 시믄 오셩 부수지 맙서..
번역: 조금만 있으면 오셔서 부수지 마세요. 예. 예뻐요. 예뻐요... 예뻐...

--- 표준어 → 제주 방언 번역 ---
원문: [표준] 좋은 아침입니다~ 여러분 주말 잘 쉬셨나요?
번역: 좋은 아침이우 다게? 여러분 주말 잘 쉬셨수과???게?게.게? 게.게.이.게마씀게.예?게양.게게. 게. 게 난게. 양.게 주말 잘 쉰 게양?게 양. 게양. 게난게. 예?게,게. 주말 잘도 쉰게마씸게.양. 양?게 게.이?게 주말게게?예?양? 주말게 잘 쉬어 수다게? 양? 게양게.마씀.게 양?양. 주말게 주말이 잘 쉬엄수과게? 예? 양. 주말이게?양 주말게 쉬어수광?게게양?? 게?게마게?? 양 양.양? 양게양게?이. 게 양. 양게게 양게.?게이.예게..게양 양.이 양.예.게이양.양 양? 양,게 양,양.이양?양게게이 양?.게 .게. 겡이. 양, 양.마씸.게,양? 게 양? 주말 잘게 쉬엄서양?마씀?게 마심.게 마씀. 게게. 허영게. 이 주말게양,게?마씸?게만게. 일 주일게게예? 주말 게게? 주말이 양게 주말 게 쉬 엄수광게?. 게??이??예게? 일주일 잘 쉬 엄 수광??.?? 겅허여.게예.? 양양.?이 양 양?? 예게.

--- 복원 태스크 ---
원문: [복원] 교육 지원을 받을때 차질이 없도록 진행해주시면 감사하겠습니다~
복원: 교육 지원을 받을때 차질이 없도록 진행해주시면 감사하겠습니다~~.

